# Lab 2 — Ridge, Lasso and Cross-Validation
**Machine Learning I · PEU-CD 2026 · ENEI**

Companion to `tutorial.pdf`. Conventions: Lecture 3's ridge is `Ridge(alpha=lam)`; Lecture 3's lasso
$\tfrac12\|y-X\beta\|^2+\lambda\|\beta\|_1$ is `Lasso(alpha=lam/N)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.linear_model import Ridge, Lasso, LogisticRegression, lasso_path
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_squared_error, accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, roc_auc_score)
np.set_printoptions(precision=4, suppress=True)

X_raw, y_raw = load_diabetes(return_X_y=True)
N, p = X_raw.shape
X = (X_raw - X_raw.mean(0)) / X_raw.std(0)      # standardized columns
y = y_raw - y_raw.mean()                         # centred response -> no intercept needed
print(X.shape, "column means ~0:", np.allclose(X.mean(0), 0), " column sds = 1:", np.allclose(X.std(0), 1))

## 1. Ridge from the formula
### Task 1 — closed form

In [ ]:
def ridge_closed_form(X, y, lam):
    # TODO: (X^T X + lam I)^{-1} X^T y, via np.linalg.solve
    ...

for lam in [0.1, 1, 10, 100]:
    mine = ridge_closed_form(X, y, lam)
    sk = Ridge(alpha=lam, fit_intercept=False).fit(X, y).coef_
    print(f"lam={lam:6}: max |diff| = {np.abs(mine - sk).max():.2e}")
    assert np.allclose(mine, sk, atol=1e-8)

### Task 2 — the SVD picture

In [ ]:
U, d, Vt = np.linalg.svd(X, full_matrices=False)
lam = 10
# TODO: shrinkage factors d^2/(d^2+lam); y_hat via the SVD formula; compare with X @ ridge_closed_form(X, y, lam)
shrink = ...
yhat_svd = ...
print("SVD formula matches closed form:", np.allclose(yhat_svd, X @ ridge_closed_form(X, y, lam)))
# TODO: which direction is shrunk most? plot df(lambda) on a log axis; confirm df(0) = 10


### Task 3 — the ridge path

In [ ]:
# TODO: coefficient paths for lams (log axis). Then explain, from the SVD formula, why no path crosses zero.


## 2. Lasso from the formula
### Task 4 — soft thresholding

In [ ]:
def soft(b, lam):
    # TODO: sign(b) * max(|b| - lam, 0), vectorized
    ...

assert np.allclose(soft([3, 0.5, -2], 1), [2, 0, -1])
assert np.allclose(np.array([3, 0.5, -2]) / 2, [1.5, 0.25, -1])

### Task 5 — coordinate descent

In [ ]:
def lasso_cd(X, y, lam, iters=20000, tol=1e-10):
    n, p = X.shape
    beta = np.zeros(p)
    r = y.copy()                           # full residual y - X beta
    for _ in range(iters):                 # stop early once no coordinate moves more than tol
        for j in range(p):
            r_j = r + X[:, j] * beta[j]    # partial residual with coordinate j removed
            # TODO: coordinate update: soft-threshold x_j' r_j, divide by ||x_j||^2; then update r and beta[j]
            b_new = ...
            r = ...
            beta[j] = b_new
    return beta

for lam in [50, 200, 800]:
    mine = lasso_cd(X, y, lam)
    sk = Lasso(alpha=lam / N, fit_intercept=False, max_iter=100000, tol=1e-10).fit(X, y).coef_
    print(f"lam={lam}: max|diff|={np.abs(mine - sk).max():.1e}  zeros={int((np.abs(mine) < 1e-8).sum())}")
    assert np.allclose(mine, sk, atol=1e-4)
# TODO: verify the KKT conditions |x_j' r| = lam (active) and <= lam (inactive) on one solution


### Task 6 — the lasso path and $\lambda_{\max}$

In [ ]:
# TODO: lam_max = ||X^T y||_inf; show lasso_cd returns zeros above it and not below it
lam_max = ...
# TODO: lasso_path (note: its alphas are lam/N) next to the ridge path; list the order in which features enter


## 3. Choosing lambda
### Task 7 — cross-validation with error bars

In [ ]:
grid = np.logspace(-1, 3.2, 40)
kf = KFold(10, shuffle=True, random_state=155)
# TODO: for each lam, a Pipeline(StandardScaler, Lasso) evaluated by cross_val_score; record mean, SE, and #nonzero
# TODO: plot with error bars; mark lam_min and the one-SE choice (largest lam with CV <= min + SE)


### Task 8 — optimism, measured

In [ ]:
# TODO: sigma^2 from OLS residuals (divide by N-p-1). For lam in [0.1,1,10,100,1000]: train MSE, 10-fold CV MSE,
# their difference, and 2*sigma^2*df(lam)/N side by side.


## 4. Regularized logistic regression
### Task 9 — separable data has no MLE

In [ ]:
rng = np.random.default_rng(155)
A = rng.normal([-2, 0], 1, (30, 2)); B = rng.normal([2, 0], 1, (30, 2))
Xs = np.vstack([A, B]); ys = np.r_[np.zeros(30), np.ones(30)]
# TODO: fit LogisticRegression(penalty=None, max_iter=10000); print ||w|| and the range of fitted probabilities
# TODO: for C in [1e3,1e2,10,1,0.1] plot ||w|| vs C (log axis) and explain the shape


### Task 10 — a real classifier, evaluated properly

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True); yc = 1 - yc      # malignant = 1
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=155, stratify=yc)
# TODO: Pipeline(StandardScaler, LogisticRegression) with C chosen by 5-fold GridSearchCV on the training set
# TODO: accuracy, precision, recall, F1, confusion matrix at threshold 0.5
# TODO: ROC curve; AUC via roc_auc_score AND as the fraction of (pos, neg) pairs ranked correctly; assert equal
# TODO: with c_FP=1, c_FN=10: tau* = c_FP/(c_FP+c_FN); confusion matrix and expected cost at 0.5 and at tau*


## 5. Exercises — see `tutorial.pdf`, Section 6